Setup

In [19]:
from pathlib import Path
import sys

import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataloader import create_dataloaders
from src.models.resnet50 import (
    create_resnet50,
    unfreeze_resnet_layer4,
)
from src.training.trainer import train_model
from src.utils.seed import set_seed

set_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


Load same shared train/validation data:

In [20]:
loaders = create_dataloaders(
    dataset_root=PROJECT_ROOT / "data/raw/garbage-dataset",
    manifest_dir=PROJECT_ROOT / "data/manifests",
    batch_size=32,
    num_workers=0,
)

train_loader = loaders["train_loader"]
val_loader = loaders["val_loader"]
class_to_idx = loaders["class_to_idx"]

Recreate ResNet50 and load Experiment 1

In [21]:
model = create_resnet50(
    num_classes=10,
    pretrained=True,
    freeze_backbone=True,
)

checkpoint_path = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "resnet50_frozen_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)

print(
    "Loaded frozen model from epoch:",
    checkpoint["epoch"],
)

Loaded frozen model from epoch: 9


In [22]:
model = unfreeze_resnet_layer4(model)

Check trainable parameters

In [23]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.2f}%"
)

Total parameters:     23,528,522
Trainable parameters: 14,985,226
Trainable percentage: 63.69%


In [24]:
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name)

layer4.0.conv1.weight
layer4.0.bn1.weight
layer4.0.bn1.bias
layer4.0.conv2.weight
layer4.0.bn2.weight
layer4.0.bn2.bias
layer4.0.conv3.weight
layer4.0.bn3.weight
layer4.0.bn3.bias
layer4.0.downsample.0.weight
layer4.0.downsample.1.weight
layer4.0.downsample.1.bias
layer4.1.conv1.weight
layer4.1.bn1.weight
layer4.1.bn1.bias
layer4.1.conv2.weight
layer4.1.bn2.weight
layer4.1.bn2.bias
layer4.1.conv3.weight
layer4.1.bn3.weight
layer4.1.bn3.bias
layer4.2.conv1.weight
layer4.2.bn1.weight
layer4.2.bn1.bias
layer4.2.conv2.weight
layer4.2.bn2.weight
layer4.2.bn2.bias
layer4.2.conv3.weight
layer4.2.bn3.weight
layer4.2.bn3.bias
fc.weight
fc.bias


## Use a much smaller learning rate

In [25]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters(),
    ),
    lr=0.0001,
    weight_decay=0.0001,
)

Fine-tune for 5 epochs initially

In [26]:
finetune_checkpoint_path = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
    / "resnet50_layer4_finetuned_best.pth"
)

finetune_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=5,
    checkpoint_path=finetune_checkpoint_path,
)


Epoch 1/5


Training:   0%|          | 0/433 [00:00<?, ?it/s]

Validation:   0%|          | 0/93 [00:00<?, ?it/s]

Train Loss: 0.1135 | Train Acc: 0.9625
Val Loss:   0.1333 | Val Acc:   0.9612
Time: 200.6s
Best model saved (val_loss=0.1333)

Epoch 2/5


Training:   0%|          | 0/433 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/93 [00:00<?, ?it/s]

Train Loss: 0.0253 | Train Acc: 0.9921
Val Loss:   0.1336 | Val Acc:   0.9632
Time: 145.4s

Epoch 5/5


Training:   0%|          | 0/433 [00:00<?, ?it/s]

Validation:   0%|          | 0/93 [00:00<?, ?it/s]

Train Loss: 0.0177 | Train Acc: 0.9949
Val Loss:   0.1296 | Val Acc:   0.9659
Time: 144.7s


Load the best fine-tuned checkpoint and run exactly the same validation metric calculation

In [29]:
from src.evaluation.metrics import (
    collect_predictions,
    calculate_classification_metrics,
)

In [31]:
class_names = [
    class_name
    for class_name, index in sorted(
        class_to_idx.items(),
        key=lambda item: item[1],
    )
]

print(class_names)

['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']


In [33]:
checkpoint = torch.load(
    finetune_checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Best fine-tuning epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Validation accuracy:", checkpoint["val_accuracy"])

y_true, y_pred, y_prob = collect_predictions(
    model=model,
    dataloader=val_loader,
    device=device,
)

metrics, report, matrix = calculate_classification_metrics(
    y_true=y_true,
    y_pred=y_pred,
    class_names=class_names,
)

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

Best fine-tuning epoch: 3
Best validation loss: 0.12801878098096803
Validation accuracy: 0.9659244264507423
accuracy: 0.9659
macro_precision: 0.9589
macro_recall: 0.9584
macro_f1: 0.9584
weighted_f1: 0.9660
